In [5]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchreid
from torchreid.utils import FeatureExtractor
from tqdm import tqdm
import numpy as np
#---------------------------------------
# Configuration and Dataset Preparation
#---------------------------------------
# Replace with your dataset root directory and dataset name
dataset_root = 'dataset'
dataset_name = 'market1501'  # or your dataset name
batch_size = 64

# Build dataset and dataloaders using torchreid's data manager
datamanager = torchreid.data.ImageDataManager(
    root=dataset_root,
    sources=dataset_name,
    height=256,
    width=128,
    batch_size_train=batch_size,
    batch_size_test=batch_size,
    transforms=['random_flip', 'random_crop'],
    combineall=False,
)

query_loader = datamanager.test_loader
gallery_loader = datamanager.test_loader

#------------------------------------------------------------
# Loading and Preparing the Torchreid Model
#------------------------------------------------------------
# Example: OSNet_x1_0 pre-trained on ImageNet, with random classifier head
torchreid_model = torchreid.models.build_model(
    name='osnet_x1_0',
    num_classes=datamanager.num_train_pids,
    loss='softmax',
    pretrained=True
)
torchreid_model = torchreid_model.cuda()
torchreid_model.eval()

# Optionally load a trained model checkpoint
# torchreid.utils.load_pretrained_weights(torchreid_model, 'path_to_checkpoint.pth')

#------------------------------------------------------------
# Loading and Preparing the Soldier Reid Model
#------------------------------------------------------------
# Assume soldier_reid_model is a PyTorch model with the same interface
# (i.e., soldier_reid_model(inputs) returns feature embeddings)
# Replace this block with your actual model loading code
soldier_reid_model = ...  # e.g., torch.load('soldier_reid_model.pth')
soldier_reid_model = soldier_reid_model.cuda()
soldier_reid_model.eval()

#------------------------------------------------------------
# Feature Extraction Function
#------------------------------------------------------------
def extract_features(model, loader):
    all_feats = []
    all_pids = []
    all_camids = []
    with torch.no_grad():
        for data in tqdm(loader, desc='Extracting features'):
            imgs, pids, camids = data
            imgs = imgs.cuda()
            feats = model(imgs)
            # Ensure feats is normalized or handled as required
            # Some models output raw features that may need L2 normalization
            if isinstance(feats, (list, tuple)):
                feats = feats[0]
            feats = nn.functional.normalize(feats, dim=1, p=2)
            all_feats.append(feats.cpu())
            all_pids.extend(pids)
            all_camids.extend(camids)
    all_feats = torch.cat(all_feats, dim=0)
    return all_feats, all_pids, all_camids

#------------------------------------------------------------
# Evaluation Metrics
#------------------------------------------------------------
def evaluate(query_feats, query_pids, query_camids, gallery_feats, gallery_pids, gallery_camids):
    # This is a simplified evaluation function.
    # Typically, you would compute distance matrix, then compute CMC and mAP.
    # Torchreid has utility functions for this. For example:
    from torchreid.metrics import compute_distance_matrix, evaluate_rank
    distmat = compute_distance_matrix(query_feats, gallery_feats, metric='euclidean')
    distmat = distmat.numpy()
    q_pids = np.array(query_pids)
    g_pids = np.array(gallery_pids)
    q_camids = np.array(query_camids)
    g_camids = np.array(gallery_camids)
    
    cmc, mAP, _, _ = evaluate_rank(distmat, q_pids, g_pids, q_camids, g_camids, use_metric_cuhk03=False)
    return cmc[0], mAP  # Return rank-1 accuracy and mAP

#------------------------------------------------------------
# Extracting Features and Evaluating Torchreid Model
#------------------------------------------------------------
query_feats_tr, query_pids_tr, query_camids_tr = extract_features(torchreid_model, query_loader)
gallery_feats_tr, gallery_pids_tr, gallery_camids_tr = extract_features(torchreid_model, gallery_loader)

torchreid_rank1, torchreid_mAP = evaluate(
    query_feats_tr, query_pids_tr, query_camids_tr,
    gallery_feats_tr, gallery_pids_tr, gallery_camids_tr
)

print("Torchreid Model Performance:")
print("Rank-1: {:.2%}".format(torchreid_rank1))
print("mAP: {:.2%}".format(torchreid_mAP))

#------------------------------------------------------------
# Extracting Features and Evaluating Soldier Reid Model
#------------------------------------------------------------
query_feats_soldier, query_pids_soldier, query_camids_soldier = extract_features(soldier_reid_model, query_loader)
gallery_feats_soldier, gallery_pids_soldier, gallery_camids_soldier = extract_features(soldier_reid_model, gallery_loader)

soldier_rank1, soldier_mAP = evaluate(
    query_feats_soldier, query_pids_soldier, query_camids_soldier,
    gallery_feats_soldier, gallery_pids_soldier, gallery_camids_soldier
)

print("Soldier Reid Model Performance:")
print("Rank-1: {:.2%}".format(soldier_rank1))
print("mAP: {:.2%}".format(soldier_mAP))

#------------------------------------------------------------
# Compare Models
#------------------------------------------------------------
print("Comparison:")
print(f"Torchreid - Rank-1: {torchreid_rank1:.2%}, mAP: {torchreid_mAP:.2%}")
print(f"Soldier - Rank-1: {soldier_rank1:.2%}, mAP: {soldier_mAP:.2%}")

if torchreid_rank1 > soldier_rank1:
    print("Torchreid model has higher Rank-1 accuracy.")
else:
    print("Soldier model has higher or equal Rank-1 accuracy.")

if torchreid_mAP > soldier_mAP:
    print("Torchreid model has higher mAP.")
else:
    print("Soldier model has higher or equal mAP.")


Building train transforms ...
+ resize to 256x128
+ random flip
+ random crop (enlarge to 288x144 and crop 256x128)
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
Building test transforms ...
+ resize to 256x128
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
=> Loading train (source) dataset
=> Loaded Market1501
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |    80 |      630 |         8
  query    |    60 |      237 |         8
  gallery  |    72 |      246 |         8
  ----------------------------------------
=> Loading test (target) dataset
=> Loaded Market1501
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |    80 |      630 |         8
  query    |    60 |      237 |         8
  gal

AssertionError: Torch not compiled with CUDA enabled

In [1]:
import torch
print(torch.cuda.is_available())
x = torch.randn(5,5).cuda()
print(x)


True
tensor([[-2.2289e-01,  2.8508e-01,  4.4121e-02,  8.0512e-01,  6.8394e-01],
        [ 3.1713e-01,  2.4119e-01, -1.0821e+00, -1.7630e+00,  2.2076e-01],
        [ 7.3290e-01,  1.6003e-01, -1.4251e-03, -4.8950e-01,  2.8135e-01],
        [ 1.4074e+00,  5.7265e-01,  5.8919e-01, -6.9194e-01,  6.2811e-01],
        [-7.3668e-01,  1.8571e-01, -1.1870e-01,  3.9852e-01, -8.2613e-01]],
       device='cuda:0')


In [2]:
import torch
import torchreid

#-----------------------------------------------------
# Step 1: Build the Data Manager
#-----------------------------------------------------
# Here we assume you want to evaluate on 'market1501'.
# The root directory should contain the dataset folder.
# For Market1501, it usually has:
# market1501/
#     bounding_box_train/
#     bounding_box_test/
#     query/
datamanager = torchreid.data.ImageDataManager(
    root='dataset',       # Path to the folder containing the dataset
    sources='market1501',           # Source dataset for training (if needed)
    targets='market1501',           # Target dataset for evaluation
    height=256,
    width=128,
    batch_size_train=32,
    batch_size_test=100,
    transforms='random_flip',       # Apply basic transformations
    combineall=False,
    workers=4
)

#-----------------------------------------------------
# Step 2: Build and Load the Model
#-----------------------------------------------------
# Suppose your model is built as follows:
model = torchreid.models.build_model(
    name='osnet_x1_0',                         # Example model architecture
    num_classes=datamanager.num_train_pids,    # Number of training person IDs
    loss='softmax',
    pretrained=True                            # Load imagenet pretraining
)

model = model.cuda()

# Load the trained weights (model.pth is your checkpoint)
torchreid.utils.load_pretrained_weights(model, 'log/osnet_x1_0_market1501_softmax_cosinelr/model/model.pth.tar-20')

# You don't necessarily need an optimizer or scheduler if you're just evaluating,
# but Torchreid's engines typically require them. Here we set them up anyway.
optimizer = torchreid.optim.build_optimizer(
    model,
    optim='adam',
    lr=0.0003
)
scheduler = torchreid.optim.build_lr_scheduler(
    optimizer,
    lr_scheduler='single_step',
    stepsize=20
)

#-----------------------------------------------------
# Step 3: Evaluate the Model
#-----------------------------------------------------
# Torchreid provides built-in engines for training and evaluation.
# If you just want to evaluate (test-only), you can use an Image-based engine.
engine = torchreid.engine.ImageSoftmaxEngine(
    datamanager,
    model,
    optimizer,
    scheduler=scheduler,
    label_smooth=True
)

# Run evaluation on the target dataset. Setting test_only=True
# means it won't do any training, just evaluation.
engine.run(
    test_only=True,
    dist_metric='euclidean',    # distance metric for evaluation
    normalize_feature=True,     # normalize features before computing distance
    ranks=[1, 5, 10, 20],       # which Rank-k accuracies to print
    rerank=False                # whether to use re-ranking
)


Building train transforms ...
+ resize to 256x128
+ random flip
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
Building test transforms ...
+ resize to 256x128
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
=> Loading train (source) dataset
=> Loaded Market1501
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |    80 |      630 |         8
  query    |    60 |      237 |         8
  gallery  |    72 |      246 |         8
  ----------------------------------------
=> Loading test (target) dataset
=> Loaded Market1501
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |    80 |      630 |         8
  query    |    60 |      237 |         8
  gallery  |    72 |      246 |         8
  -------------

/home/wins054/Documents/Project/CameraProject/deep-person-reid/torchreid/data/datasets/image/market1501.py:37: UserWarning: The current data structure is deprecated. Please put data folders such as "bounding_box_train" under "Market-1501-v15.09.15".
  warnings.warn(
/home/wins054/Documents/Project/CameraProject/deep-person-reid/torchreid/models/osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.seriali

Done, obtained 237-by-512 matrix
Extracting features from gallery set ...
Done, obtained 246-by-512 matrix
Speed: 0.0407 sec/batch
Normalzing features with L2 norm ...
Computing distance matrix with metric=euclidean ...
Computing CMC and mAP ...
** Results **
mAP: 9.9%
CMC curve
Rank-1  : 7.7%
Rank-5  : 20.9%
Rank-10 : 30.8%
Rank-20 : 39.6%


In [1]:
import sys
import os

# Add project root to PYTHONPATH if needed
sys.path.append('.')

from fastreid.config import get_cfg
from fastreid.engine import DefaultPredictor
from fastreid.data.datasets import register_market1501
from fastreid.evaluation import inference_on_dataset, build_reid_test_loader
from fastreid.utils.checkpoint import Checkpointer

#-----------------------------------------------------
# Step 1: Register Market1501 Dataset
#-----------------------------------------------------
# Make sure that the dataset directory structure is correct.
# The root directory should contain 'bounding_box_train', 'bounding_box_test', 'query'.
dataset_root = "dataset/market1501"
register_market1501('market1501', root=dataset_root)

#-----------------------------------------------------
# Step 2: Load Configuration
#-----------------------------------------------------
# Use the provided config file from the SOLIDER-REID repo that matches the Market1501 experiment.
cfg = get_cfg()
cfg.merge_from_file("/home/wins054/Documents/Project/CameraProject/SOLIDER-ReID/configs/market/swin_base.yml")  # Adjust path if needed

# Set the path to the model weights (trained on Market1501)
cfg.MODEL.WEIGHTS = "/home/wins054/Documents/Project/CameraProject/SOLIDER-ReID/weights/swin_base_msmt17.pth"
cfg.MODEL.DEVICE = "cuda"  # or "cpu" if no GPU is available

# If batch size or other test parameters need adjusting, you can do so here.
# For example:
# cfg.TEST.IMS_PER_BATCH = 128

#-----------------------------------------------------
# Step 3: Create the Predictor and Loader
#-----------------------------------------------------
predictor = DefaultPredictor(cfg)
val_loader = build_reid_test_loader(cfg, "market1501")

#-----------------------------------------------------
# Step 4: Run Inference and Evaluate
#-----------------------------------------------------
res = inference_on_dataset(predictor.model, val_loader, None)
print("Evaluation Results on Market1501:", res)


ModuleNotFoundError: No module named 'fastreid'